# Lowercase Letters CNN (a-z)

MNIST 수업에서 썼던 CNN 구조를 그대로 가져와서, 이번엔 숫자 대신 영어 소문자 26개를 분류하도록 확장해본 탐구.

In [ ]:
import keras
from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Input
import numpy as np
import string

LABELS = list(string.ascii_lowercase)  # a-z, same order the dataset was labeled with

**데이터 준비: 왜 EMNIST 같은 실제 손글씨 데이터셋을 안 썼는가**

MNIST는 `keras.datasets.mnist`가 알아서 다운로드해주지만, 진짜 손글씨 소문자 데이터셋(EMNIST 등)은 이 환경에서 접근 가능한 도메인이 github.com / pypi.org 정도로 제한돼 있어서 받을 수가 없었다 (NIST 원본 서버, Google Drive, TFDS의 GCS 버킷 전부 연결 실패). 그래서 시스템에 깔려 있는 폰트 약 230개로 a-z를 직접 렌더링해서 데이터셋을 만들었다. 폰트마다 크기·위치·회전을 랜덤하게 주고 노이즈도 살짝 섞어서, 손글씨는 아니지만 그럭저럭 다양한 26-클래스 이미지 분류 문제가 되도록 했다.

In [1]:
# Build a synthetic a-z dataset by rendering system fonts, since every real
# handwritten-letter dataset host (NIST, Google Drive, TFDS's GCS bucket) is
# unreachable from this sandbox -- only github.com and pypi.org are open here.
import glob, random
from PIL import Image, ImageDraw, ImageFont

random.seed(42)
np.random.seed(42)

IMG_SIZE = 28
CANVAS = 64  # draw big, downscale after -- smoother edges than drawing straight at 28px

FONT_PATHS = sorted(set(
    glob.glob("/usr/share/fonts/**/*.ttf", recursive=True) +
    glob.glob("/usr/share/fonts/**/*.otf", recursive=True) +
    glob.glob("/usr/share/texmf/**/*.otf", recursive=True)
))
BAD_HINTS = ("cjk", "symbol", "math", "emoji", "braille", "nerd", "wingding", "webding")
FONT_PATHS = [f for f in FONT_PATHS if not any(b in f.lower() for b in BAD_HINTS)]
print(f"usable fonts: {len(FONT_PATHS)}")


def render_letter(letter, font_path, font_size, dx, dy, angle):
    """Draw one letter and return a 28x28 uint8 array (white glyph on black, like MNIST)."""
    img = Image.new("L", (CANVAS, CANVAS), color=0)
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype(font_path, font_size)
    except Exception:
        return None

    # center the glyph, then nudge it a bit so the model doesn't just learn
    # "the letter is always dead center"
    bbox = draw.textbbox((0, 0), letter, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (CANVAS - w) / 2 - bbox[0] + dx
    y = (CANVAS - h) / 2 - bbox[1] + dy
    draw.text((x, y), letter, fill=255, font=font)

    if angle != 0:
        img = img.rotate(angle, resample=Image.BILINEAR, fillcolor=0)

    img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    arr = np.array(img, dtype=np.uint8)

    # some fonts just don't ship a glyph for a given letter and render blank -- skip those
    if arr.max() < 40:
        return None
    return arr


def build_dataset(fonts, samples_per_font_per_letter=3):
    xs, ys = [], []
    for font_path in fonts:
        for label, letter in enumerate(LABELS):
            for _ in range(samples_per_font_per_letter):
                font_size = random.randint(30, 44)
                dx, dy = random.randint(-3, 3), random.randint(-3, 3)
                angle = random.uniform(-12, 12)
                arr = render_letter(letter, font_path, font_size, dx, dy, angle)
                if arr is None:
                    continue
                if random.random() < 0.5:  # light noise for a bit of pen/anti-aliasing variety
                    noise = np.random.normal(0, 8, arr.shape)
                    arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
                xs.append(arr)
                ys.append(label)
    return np.array(xs, dtype=np.uint8), np.array(ys, dtype=np.uint8)

usable fonts: 231


In [2]:
random.shuffle(FONT_PATHS)
x_all, y_all = build_dataset(FONT_PATHS, samples_per_font_per_letter=3)
print(f"total samples: {len(x_all)}")

# 85/15 split, roughly the same ratio MNIST ships with
idx = np.random.permutation(len(x_all))
x_all, y_all = x_all[idx], y_all[idx]
n_test = int(len(x_all) * 0.15)
x_test_raw, y_test_raw = x_all[:n_test], y_all[:n_test]
x_train_raw, y_train_raw = x_all[n_test:], y_all[n_test:]
print(f"train: {x_train_raw.shape}, test: {x_test_raw.shape}")

total samples: 18013
train: (15312, 28, 28), test: (2701, 28, 28)


**데이터 전처리** (MNIST.ipynb과 완전히 동일한 방식: reshape -> float32/255 -> one-hot)

In [ ]:
BATCH_SIZE = 128
NUM_CLASSES = 26      # a-z instead of MNIST's 0-9
NUM_EPOCHS = 8        # a bit more than the MNIST notebook since 26 classes is a harder problem

x_train = x_train_raw.reshape(x_train_raw.shape[0], 28, 28, 1)
x_test = x_test_raw.reshape(x_test_raw.shape[0], 28, 28, 1)

x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255

y_train = keras.utils.to_categorical(y_train_raw, NUM_CLASSES)
y_test = keras.utils.to_categorical(y_test_raw, NUM_CLASSES)

**CNN(합성곱 신경망) 모델 구조 설계** -- MNIST.ipynb과 층 구성이 완전히 동일하다. 마지막 Dense층 출력 개수만 10 -> 26으로 바뀐다.

In [ ]:
model = Sequential()
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))

model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(NUM_CLASSES, activation='softmax'))
model.summary()

Model: "sequential"
+---------------------------------+------------------------+---------------+
| Layer (type)                    | Output Shape           |       Param # |
+---------------------------------+------------------------+---------------+
| conv2d (Conv2D)                 | (None, 26, 26, 32)     |           320 |
| conv2d_1 (Conv2D)                | (None, 24, 24, 64)     |        18,496 |
| max_pooling2d (MaxPooling2D)    | (None, 12, 12, 64)     |             0 |
| dropout (Dropout)               | (None, 12, 12, 64)     |             0 |
| flatten (Flatten)               | (None, 9216)           |             0 |
| dense (Dense)                   | (None, 128)            |     1,179,776 |
| dropout_1 (Dropout)             | (None, 128)            |             0 |
| dense_1 (Dense)                 | (None, 26)             |         3,354 |
+---------------------------------+------------------------+---------------+
Total params: 1,201,946 (4.59 MB)
Trainable params: 1,2

**모델 컴파일 / 학습 진행**

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print('Training the CNN on synthetic lowercase letters...')
history = model.fit(x_train, y_train, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS,
                     verbose=2, validation_split=0.1)

Training the CNN on synthetic lowercase letters...
Epoch 1/8
108/108 - 22s - 200ms/step - accuracy: 0.3864 - loss: 2.1039 - val_accuracy: 0.7565 - val_loss: 0.9716
Epoch 2/8
108/108 - 22s - 200ms/step - accuracy: 0.7146 - loss: 0.9701 - val_accuracy: 0.8949 - val_loss: 0.4702
Epoch 3/8
108/108 - 22s - 200ms/step - accuracy: 0.8229 - loss: 0.6218 - val_accuracy: 0.9217 - val_loss: 0.3366
Epoch 4/8
108/108 - 22s - 200ms/step - accuracy: 0.8616 - loss: 0.4831 - val_accuracy: 0.9341 - val_loss: 0.2917
Epoch 5/8
108/108 - 22s - 200ms/step - accuracy: 0.8869 - loss: 0.4035 - val_accuracy: 0.9380 - val_loss: 0.2713
Epoch 6/8
108/108 - 22s - 200ms/step - accuracy: 0.9018 - loss: 0.3474 - val_accuracy: 0.9471 - val_loss: 0.2388
Epoch 7/8
108/108 - 22s - 200ms/step - accuracy: 0.9126 - loss: 0.3057 - val_accuracy: 0.9458 - val_loss: 0.2282
Epoch 8/8
108/108 - 22s - 200ms/step - accuracy: 0.9132 - loss: 0.2899 - val_accuracy: 0.9517 - val_loss: 0.2241


**모델 성능 평가**

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print('Final test accuracy: %.2f%%' % (score[1] * 100))
print('Final test loss: %.4f' % score[0])

Final test accuracy: 95.30%
Final test loss: 0.2052


**결과 확인: 임의의 테스트 이미지 25장에 대한 예측**

MNIST 노트북 마지막 셀(랜덤 25장 시각화)과 같은 형태로, 이번엔 예측값까지 같이 표시해봤다.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for i in range(25):
    idx = np.random.randint(0, len(x_test))
    pred_label = LABELS[np.argmax(model.predict(x_test[idx:idx+1], verbose=0))]
    true_label = LABELS[np.argmax(y_test[idx])]
    plt.subplot(5, 5, i + 1)
    plt.imshow(x_test[idx].reshape(28, 28), cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    plt.title(f'pred: {pred_label} (true: {true_label})', color=color, fontsize=9)
    plt.axis('off')
plt.suptitle('Random Test Predictions', fontsize=16)
plt.tight_layout()
plt.show()

**한계 및 다음에 시도해볼 것**

이 데이터셋은 진짜 손글씨가 아니라 폰트로 찍어낸 글자라서, 실제 사람 손글씨(그림판에 마우스로 그린 것 등)에는 MNIST 때만큼 잘 일반화되지 않을 가능성이 크다. 다음 단계로는 (1) 네트워크가 열린 환경에서 실제 EMNIST를 받아 이 모델과 정확도를 비교하거나, (2) 그림판 입력을 받아서 직접 테스트해보는 것을 시도해볼 수 있다.